## Data Cleaning & Preprocessing

This notebook cleans and prepocesses the data using the api built from src

**Run before notebook to build data from zip:** `python scripts/build_data.py all` (may take a few minutes)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from preprocess import OUT
frames = pd.read_csv(OUT / "manifest.csv")
frames.shape

#### 1. Check for missing or corrupted images and annotations

In [ ]:
frames[[c for c in frames.columns if c.startswith("flag_")]].sum()

In [ ]:
frames.usable.sum(), len(frames)

#### 2. Fix the corner ordering

We must reorient the labels so that the top left label of the paper always corresponds to the top left of the image

In [ ]:
pd.crosstab(frames.bg_name, frames.rot_class)

In [ ]:
frames.doc_corner_at_slot0.value_counts()

#### 3. Sample frames from each video so we do not use too many nearly identical images

In [ ]:
# Find how close on average two frames are to each other
frames.center_step_px.median().round(1)

In [ ]:
# Sample every third frame
from preprocess import subsample
filtered_frames = subsample(frames, stride=3)
len(frames), len(filtered_frames)

### 4. Normalize the paper's corners to a value 0-1

In [ ]:
from preprocess import TARGET_COLS
y = filtered_frames[TARGET_COLS].to_numpy("float32")
y.shape, round(float(y.min()), 3), round(float(y.max()), 3)

In [ ]:
# denormalize back to pixel coordinates so we can later calc error
from preprocess import denormalise
denormalise(y[:1])[0].round(1)

#### 5. Resize the images to the same size and normalize the pixel values

In [ ]:
from preprocess import load_frame
img = load_frame(filtered_frames.image_path[0], (224, 224), "rgb")
img.shape, img.dtype

In [ ]:
# format in tensor format
from augment import to_tensor_chw
x = to_tensor_chw(img)
x.shape, round(float(x.mean()), 3), round(float(x.std()), 3)

In [ ]:
q = denormalise(y[:1])[0] / [1920 / 224, 1080 / 224]

# plot bbox to ensure proper denormalization
plt.imshow(img); plt.plot(*np.vstack([q, q[:1]]).T, "r-", lw=2); plt.axis("off");

#### 6. Augment Data (brightness, blur, rotation, perspective)

In [ ]:
# Augment image by moving its corners through jittering
from augment import augment
rng = np.random.default_rng(0)
img2, q2 = augment(img.copy(), q.copy(), rng)
#display how far the corners have moved
np.linalg.norm(q2 - q, axis=1).round(1)

In [ ]:
# Create augmentation by jittering image to offset its position
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
for a, (im, qq) in zip(ax, [(img, q), (img2, q2)]):
    a.imshow(im); a.plot(*np.vstack([qq, qq[:1]]).T, "r-", lw=2); a.axis("off")

#### 7. Splits — grouped by video so near-duplicate frames cannot leak

In [ ]:
filtered_frames.split_video.value_counts()

In [ ]:
filtered_frames.groupby("video_id").split_video.nunique().max()

In [ ]:
pd.crosstab(filtered_frames.bg_name, filtered_frames.split_video)

#### 8. Model data

Option A: load entire dataset into memory

Option B: read photos from disk as the model asks for them, optionally
          adding random augment to generate more unique samples.

In [ ]:
# Option A - Splits already applied.
from dataset import load_arrays
parts, groups, _ = load_arrays("img_128x72_rgb")
X_train, y_train = parts["train"]
X_train.shape, y_train.shape

In [ ]:
# Option B - streaming and augmenting for cnn
from dataset import FrameSet
train_ds = FrameSet("train", size=(224, 224), augment=True)
len(train_ds), train_ds[0][0].shape

#### 9. Create benchmark by guessing same position for each sample (average pos)

Create a benchmark to ensure the model learned by comparing it to just guessing the average position for every sample.

In [ ]:
from baselines import main as run_baseline
run_baseline()

## Modeling, Tuning, and Final Evaluation

This section combines the modeling work into the same notebook so the full workflow is in one place.

It covers:
- verification of the exported arrays and shared split
- Ridge, MLP, grayscale CNN, and RGB CNN experiment runs
- optional color, sharpening, and PCA comparisons
- final held-out test evaluation and result review

In [ ]:
from pathlib import Path
import subprocess, json
import numpy as np, pandas as pd

PROJECT = Path.cwd()
OUT = PROJECT / "data" / "processed"
RESULTS = PROJECT / "results"

def run_py(*args):
    cmd = ["python", *map(str, args)]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

#### 10. Verify exported arrays and the shared split

Run this before training to confirm the arrays, labels, and splits line up with the modeling plan.

In [ ]:
manifest = pd.read_csv(OUT / "manifest_subset.csv")
gray_npz = np.load(OUT / "arrays" / "tab_64x36_clahe.npz", allow_pickle=True)
rgb_npz = np.load(OUT / "arrays" / "img_128x72_rgb.npz", allow_pickle=True)

summary = {
    "manifest_subset_rows": len(manifest),
    "gray_keys": list(gray_npz.keys()),
    "gray_X_shape": tuple(gray_npz["X"].shape),
    "gray_y_shape": tuple(gray_npz["y"].shape),
    "gray_dtype": str(gray_npz["X"].dtype),
    "rgb_keys": list(rgb_npz.keys()),
    "rgb_X_shape": tuple(rgb_npz["X"].shape),
    "rgb_y_shape": tuple(rgb_npz["y"].shape),
    "rgb_dtype": str(rgb_npz["X"].dtype),
    "target_cols": gray_npz["target_cols"].tolist(),
}
summary

In [ ]:
video_split = manifest[["video_id", "split_video"]].drop_duplicates()
doc_split = manifest[["model_id", "split_doc"]].drop_duplicates()

video_groups = {split: set(video_split.loc[video_split.split_video == split, "video_id"]) for split in ["train", "val", "test"]}
doc_groups = {split: set(doc_split.loc[doc_split.split_doc == split, "model_id"]) for split in ["train", "val", "test"]}

checks = {
    "train_val_video_disjoint": video_groups["train"].isdisjoint(video_groups["val"]),
    "train_test_video_disjoint": video_groups["train"].isdisjoint(video_groups["test"]),
    "val_test_video_disjoint": video_groups["val"].isdisjoint(video_groups["test"]),
    "train_val_doc_disjoint": doc_groups["train"].isdisjoint(doc_groups["val"]),
    "train_test_doc_disjoint": doc_groups["train"].isdisjoint(doc_groups["test"]),
    "val_test_doc_disjoint": doc_groups["val"].isdisjoint(doc_groups["test"]),
    "gray_rows_match_manifest": len(gray_npz["y"]) == len(manifest),
    "rgb_rows_match_manifest": len(rgb_npz["y"]) == len(manifest),
}
checks

#### 11. Run the planned model comparisons

Run these cells in order after the split checks pass. The Ridge and MLP cells include the PCA experiment, the optional sharpening experiment, and the smaller grayscale-versus-RGB comparison for a simpler model.

In [ ]:
run_py("scripts/run_ridge.py")

In [ ]:
run_py("scripts/run_mlp.py", "--compare-color", "--compare-sharpening")

In [ ]:
run_py("scripts/run_cnn.py", "--config", "configs/cnn_gray.yaml", "--overfit-test")
run_py("scripts/run_cnn.py", "--config", "configs/cnn_gray.yaml")

In [ ]:
run_py("scripts/run_cnn.py", "--config", "configs/cnn_rgb.yaml", "--overfit-test")
run_py("scripts/run_cnn.py", "--config", "configs/cnn_rgb.yaml")

#### 12. Final held-out test evaluation

This cell evaluates the selected Ridge, MLP, grayscale CNN, and RGB CNN once on the same test split and writes the comparison tables to the results folder.

In [ ]:
run_py("scripts/final_eval.py")

#### 13. Review saved results

These cells load the exported comparison tables after the training and evaluation scripts finish.

In [ ]:
comparison = pd.read_csv(RESULTS / "final_comparison.csv")
comparison

In [ ]:
by_background = pd.read_csv(RESULTS / "by_background.csv")
by_background